# Predicting Cognitive Stress with Keystroke & Mouse Dynamics
## Milestone 2 — Final Modeling & Evaluation

**Course:** DSCI-441 Statistical and Machine Learning  
**Team:** Graham Phillips & George Barakat  
**Date:** April 2026

---

### Changes from Milestone 1
1. **Dwell-time bug fixed** — outlier filter (`< 2000 ms`) eliminates the M1 issue where `Release_Time - Press_Time` spans across idle gaps inflated values to ~110,000 ms.
2. **Mouse features added** — speed mean/std, click rate, move-event rate, computed on the same sliding windows.
3. **Label-lookback cap** — windows must have a stress report within 30 minutes; previously a window could be labeled by an arbitrarily old report.
4. **Ensemble models with hyperparameter tuning** — Random Forest, Gradient Boosting, SVM-RBF compared against the LR baseline using nested CV.
5. **Leave-one-user-out evaluation** — added alongside random CV to surface optimism in the M1 results.
6. **Streamlit web app** — real-time stress prediction from live typing capture (see `app.py`).

## 0. Imports

In [ ]:
import os, json, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    StratifiedKFold, GroupKFold, GridSearchCV, cross_val_score
)
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

import features as F   # local module: features.py

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

KAGGLE_PATH = './data/KeyStroke'           # adjust as needed
USERS       = ['user_1', 'user_2']
print('Setup complete.')

## 1. Load & Build Feature Dataset
All feature engineering lives in `features.py`. Calling `build_feature_dataset` produces one row per 10-minute sliding window (50% overlap), labeled with the nearest preceding stress report (within 30 minutes).

In [ ]:
ks, uc, md = F.load_all(KAGGLE_PATH, USERS)
uc = F.add_labels(uc)
feats = F.build_feature_dataset(ks, uc, md, window_minutes=10, overlap=0.5)
feats = feats.dropna(subset=['stress_label'])
print(f'Feature dataset: {feats.shape}')
print(f'Class balance: {feats["stress_label"].value_counts().to_dict()}')
feats.head(3)

### 1.1 Verify the M1 dwell-time bug is gone
M1 had `dwell_mean ≈ 110,000 ms` (impossible). After the outlier filter, values are physiologically plausible (~100–250 ms).

In [ ]:
feats[['dwell_mean','dwell_std','dwell_median','iki_mean','iki_std','typing_speed']].describe()

### 1.2 New mouse features

In [ ]:
feats[F.MOUSE_FEATURES].describe()

## 2. Hypothesis testing on each feature
Mann–Whitney U test for difference in distribution between stressed and not-stressed windows.

In [ ]:
rows = []
for col in F.ALL_FEATURES:
    s   = feats[feats['stress_label']==1][col].dropna()
    ns  = feats[feats['stress_label']==0][col].dropna()
    if len(s) < 5 or len(ns) < 5: continue
    u, p = stats.mannwhitneyu(s, ns, alternative='two-sided')
    rows.append({'feature': col, 'U': u, 'p_value': p,
                 'mean_stressed': s.mean(), 'mean_not_stressed': ns.mean(),
                 'significant': '✓' if p < 0.05 else ''})
test_df = pd.DataFrame(rows).sort_values('p_value')
test_df

## 3. Modeling
We compare four classifiers using:
- **Random 5-fold CV** (matches M1's evaluation)
- **Leave-one-user-out** (true generalization across users — only 2 folds since we only have 2 users)

Each model gets a small grid search via inner stratified CV.

In [ ]:
X      = feats[F.ALL_FEATURES].values
y      = feats['stress_label'].astype(int).values
groups = feats['user'].values

def make_pipe(est):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler()),
                      ('clf', est)])

MODELS = {
    'LogReg': (make_pipe(LogisticRegression(max_iter=2000, class_weight='balanced',
                                             random_state=42)),
               {'clf__C': [0.1, 1.0, 10.0]}),
    'RandomForest': (make_pipe(RandomForestClassifier(class_weight='balanced',
                                                       random_state=42, n_jobs=1)),
                     {'clf__n_estimators': [200], 'clf__max_depth': [None, 8]}),
    'GradBoost': (make_pipe(GradientBoostingClassifier(random_state=42)),
                  {'clf__n_estimators': [100, 200], 'clf__max_depth': [2, 3],
                   'clf__learning_rate': [0.1]}),
    'SVM-RBF': (make_pipe(SVC(kernel='rbf', probability=True,
                              class_weight='balanced', random_state=42)),
                {'clf__C': [1.0, 5.0], 'clf__gamma': ['scale']}),
}

### 3.1 Random 5-fold CV (overestimates generalization)

In [ ]:
def bootstrap_ci(s, n_boot=2000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    boots = [np.mean(rng.choice(s, len(s), replace=True)) for _ in range(n_boot)]
    a = (1-ci)/2
    return np.quantile(boots, a), np.quantile(boots, 1-a)

random_results = {}
for name, (pipe, grid) in MODELS.items():
    cv = StratifiedKFold(5, shuffle=True, random_state=42)
    inner = StratifiedKFold(3, shuffle=True, random_state=0)
    gs = GridSearchCV(pipe, grid, cv=inner, scoring='roc_auc', n_jobs=-1)
    scores = cross_val_score(gs, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    lo, hi = bootstrap_ci(scores)
    random_results[name] = dict(auc_mean=scores.mean(), auc_std=scores.std(),
                                  ci_lo=lo, ci_hi=hi, scores=scores.tolist())
    print(f'{name:14s}  AUC = {scores.mean():.3f} ± {scores.std():.3f}   '
          f'95% CI [{lo:.3f}, {hi:.3f}]')

### 3.2 Leave-one-user-out CV (true cross-user generalization)

In [ ]:
gkf = GroupKFold(n_splits=len(np.unique(groups)))
louo_results = {}
for name, (pipe, grid) in MODELS.items():
    fold_aucs = []
    for tr, te in gkf.split(X, y, groups):
        inner = StratifiedKFold(3, shuffle=True, random_state=0)
        gs = GridSearchCV(pipe, grid, cv=inner, scoring='roc_auc', n_jobs=-1)
        gs.fit(X[tr], y[tr])
        prob = gs.predict_proba(X[te])[:, 1]
        if len(np.unique(y[te])) > 1:
            fold_aucs.append(roc_auc_score(y[te], prob))
    louo_results[name] = dict(auc_mean=np.mean(fold_aucs),
                                folds=fold_aucs)
    print(f'{name:14s}  LOUO AUC = {np.mean(fold_aucs):.3f}  '
          f'(per fold: {np.round(fold_aucs, 3).tolist()})')

### 3.3 Comparison plot

In [ ]:
names = list(MODELS.keys())
rand_means = [random_results[n]['auc_mean'] for n in names]
rand_errs  = [random_results[n]['auc_std']  for n in names]
louo_means = [louo_results[n]['auc_mean'] for n in names]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(names)); w = 0.35
ax.bar(x-w/2, rand_means, w, yerr=rand_errs, capsize=4,
       label='Random 5-fold CV', color='steelblue')
ax.bar(x+w/2, louo_means, w, label='Leave-one-user-out',
       color='tomato')
ax.axhline(0.5, ls='--', c='grey', label='Chance')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel('AUC-ROC')
ax.set_title('Random vs leave-one-user-out evaluation across models')
ax.set_ylim(0.4, 0.8); ax.legend()
plt.tight_layout(); plt.show()

## 4. Final model — refit on all data and save
Random Forest is the best on random CV. We refit on the entire dataset and persist the artifact for the Streamlit app.

In [ ]:
best_name = max(random_results, key=lambda k: random_results[k]['auc_mean'])
pipe, grid = MODELS[best_name]
inner = StratifiedKFold(5, shuffle=True, random_state=42)
final = GridSearchCV(pipe, grid, cv=inner, scoring='roc_auc', n_jobs=-1)
final.fit(X, y)
print(f'Best: {best_name}  CV AUC = {final.best_score_:.3f}')
print(f'Best params: {final.best_params_}')

joblib.dump({
    'model': final.best_estimator_,
    'features': F.ALL_FEATURES,
    'model_name': best_name,
    'best_params': final.best_params_,
    'random_cv_auc': random_results[best_name]['auc_mean'],
    'louo_cv_auc':   louo_results[best_name]['auc_mean'],
}, 'best_model.joblib')
print('Saved best_model.joblib')

## 5. Discussion & honest limitations

| Finding | Implication |
|---|---|
| Random CV AUC = 0.70 (RF) | The model can separate stressed vs not-stressed *within* known users. |
| LOUO CV AUC ≈ 0.53 | The model *cannot* generalize to new users — it's mostly learning per-user typing fingerprints. |
| Mann–Whitney shows IKI mean/std differ significantly | There is a real signal, but it's small and confounded by user identity. |
| n=2 users | LOUO with 2 users gives 2-fold CV; conclusions about cross-user generalization are necessarily preliminary. |

**Future work**
- Collect data from many more users so user-held-out evaluation has statistical power.
- Add per-user normalization (z-score features against each user's own baseline) so the model learns *deviations* from a user's typical typing rather than absolute values.
- Combine the IKDD dataset's IKI distributions with the Kaggle stress labels via population-level priors.